# Do Banks' Disclosed VaR Figures Hold Up?
## An Empirical Audit of 10-Q Backtesting Disclosures

*Research notebook mirroring the structure of the SSRN paper.*

Under the Basel III internal-models approach, large banks compute market-risk
capital from internal one-day Value-at-Risk (VaR) models and must backtest them
against 250 days of P&L under the supervisory traffic-light system. U.S.
securities regulation simultaneously forces those same banks to *publish*
average, high, and low one-day trading VaR in every 10-Q and 10-K. This
notebook performs an independent, outside audit of those disclosures for five
large U.S. dealers — JPMorgan Chase, Goldman Sachs, Morgan Stanley, Bank of
America, and Citigroup — over 2019–2024, a window spanning the COVID-19 crash,
the 2022 rates repricing, and the March 2023 regional-banking stress.

**Hypothesis.** Disclosed VaR is conservatively calibrated on average
(unconditional coverage holds), but violations cluster in high-VIX regimes
because quarterly-average disclosures adapt too slowly — so Christoffersen's
independence and conditional-coverage tests reject even where Kupiec's level
test does not.


In [ ]:
# Setup: run everything from the project root.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)

import pandas as pd
from IPython.display import Image, display

import config
config.ensure_dirs()
print('Project root:', ROOT)


## 1. Data

Three sources are combined. **Market data**: daily adjusted close prices and
log returns for the five tickers plus the VIX, via yfinance; two macro context
series (high-yield OAS, fed funds upper bound) via FRED. **Disclosures**: every
in-sample 10-Q/10-K is pulled from SEC EDGAR (submissions API, cross-checked
with full-text search), parsed with BeautifulSoup, and the trading-VaR table
extracted by regex; quarters that cannot be machine-read are linearly
interpolated and flagged. Scraping respects SEC fair-access rules (declared
User-Agent, 2-second spacing, exponential backoff).


In [ ]:
# Fetch market data (skips gracefully if files already exist on disk).
from data import fetch

if not (config.RAW_DIR / 'bank_returns.csv').exists():
    fetch.main()
returns = pd.read_csv(config.RAW_DIR / 'bank_returns.csv', index_col='date', parse_dates=True)
returns.describe().T


In [ ]:
# Scrape EDGAR VaR disclosures (slow: ~120 filings at 2s spacing).
if not (config.RAW_DIR / 'var_disclosures.csv').exists():
    import data.scrape_edgar as scrape_edgar
    sys.argv = ['scrape_edgar.py']  # avoid Jupyter's kernel argv leaking into argparse
    scrape_edgar.main()
disclosures = pd.read_csv(config.RAW_DIR / 'var_disclosures.csv', parse_dates=['period_end'])
disclosures.groupby('bank')[['var_1day_avg_mm']].describe()


The table above summarizes the disclosed one-day trading VaR panel: 24
quarters per bank, with the stated confidence level (95% for JPM/GS/MS, 99%
for BAC/C), methodology, and the interpolation flag. Note the COVID-era jump
in average VaR across all five banks during 2020.


## 2. Violation analysis

Disclosed dollar VaR is normalized into return space by dividing by a
trading-equity base (market cap × fixed per-bank trading intensity), and a
violation is a day where the absolute daily log return exceeds the threshold —
a deliberately conservative definition, flagged in the paper's caveats.


In [ ]:
from analysis import violations
violations.main()
summary = pd.read_csv(config.PROCESSED_DIR / 'violation_summary.csv')
summary[summary['period'] == 'full_sample']


In [ ]:
display(Image(str(config.FIGURES_DIR / 'violation_rates_bar.png'), width=750))


*Figure 1. Actual vs expected violation rates with 95% confidence intervals.*

Coverage is in the neighborhood of stated levels for most banks; the
interesting question is *when* violations happen, not just how often.


In [ ]:
display(Image(str(config.FIGURES_DIR / 'violation_calendar_JPM.png'), width=750))


*Figure 2. Violation clustering by calendar month (JPMorgan). March 2020 and*
*the 2022 tightening episodes dominate — the visual signature of clustering.*


## 3. Statistical backtests

**Kupiec (1995) POF**: a likelihood-ratio test of whether the observed
violation frequency matches the stated rate, with a two-tailed non-rejection
interval for the violation count. **Christoffersen (1998)**: separates
unconditional coverage (UC) from independence (IND) of the hit sequence and
combines them into conditional coverage (CC = UC + IND, χ²(2)).


In [ ]:
from analysis import christoffersen, kupiec
christoffersen.main()
kupiec.main()
christ = pd.read_csv(config.PROCESSED_DIR / 'christoffersen_results.csv')
christ[christ['sample'] == 'full_sample'].pivot(index='bank', columns='test', values='p_value')


In [ ]:
display(Image(str(config.FIGURES_DIR / 'christoffersen_pvalue_heatmap.png'), width=600))


*Figure 3. Christoffersen p-values; red cells reject H0 at 5%. Rejections are*
*concentrated in the IND and CC columns: clustering, not average level, is the*
*dominant failure mode.*


In [ ]:
display(Image(str(config.FIGURES_DIR / 'kupiec_intervals.png'), width=950))


*Figure 4. Annual violations against the Kupiec 95% non-rejection band; red*
*points fall outside the acceptable range, almost always in stress years.*


## 4. Cross-bank comparison and stress amplification


In [ ]:
from analysis import comparison
comparison.main()
pd.read_csv(config.PROCESSED_DIR / 'cross_bank_comparison.csv')


In [ ]:
display(Image(str(config.FIGURES_DIR / 'stress_amplification_heatmap.png'), width=700))


*Figure 5. Stress amplification (stress ÷ calm violation rate). A correctly*
*specified model keeps this ratio near one; quarterly-average disclosures do not.*


## 5. Model reimplementation

Three textbook 99% one-day VaR models are estimated on the same returns:
rolling 250-day historical simulation, parametric variance–covariance (normal
and Student-t with MLE degrees of freedom), and RiskMetrics EWMA (λ = 0.94).
All forecasts are strictly out-of-sample and face the identical violation
definition and Kupiec test as the disclosures.


In [ ]:
from backtest import var_models
var_models.main()
comp = pd.read_csv(config.PROCESSED_DIR / 'model_comparison.csv')
comp.pivot(index='bank', columns='model', values='violation_rate')


In [ ]:
display(Image(str(config.FIGURES_DIR / 'model_violation_comparison.png'), width=800))


*Figure 6. Violation rates: disclosed VaR vs reimplemented models. EWMA — the*
*only specification updating volatility daily — is typically closest to its 1%*
*target, suggesting the binding constraint is disclosure cadence, not modeling*
*technology.*


## 6. Conclusions

1. **Unconditional coverage is broadly defensible** once disclosed VaR is
   normalized into return space, echoing Berkowitz & O'Brien (2002) and
   Pérignon & Smith (2010).
2. **Independence fails systematically**: violations cluster in high-VIX
   regimes and conditional coverage rejects for most banks — a failure mode
   the Basel traffic-light backtest cannot see by construction.
3. **Simple public-data models are competitive** with disclosed figures,
   making disclosure-based conditional-coverage audits a cheap supervisory
   complement. Requiring daily (or monthly) VaR disclosure with dated
   exceptions would let outside monitors run this audit continuously.

**Limitations**: equity returns proxy for non-public trading P&L; the
trading-intensity normalization is an assumption; interpolated quarters are
flagged. Results speak most credibly to the *dynamics* of disclosed VaR.


## 7. Generate the SSRN-ready paper

The final step compiles the full paper PDF (and figures appendix) directly
from the processed results, so every table and headline number in the
manuscript reflects this run.


In [ ]:
import subprocess
result = subprocess.run([sys.executable, str(ROOT / 'paper' / 'generate_paper.py')],
                        capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else '')
print(result.stderr[-2000:] if result.stderr else '')
print('Paper at:', config.PAPER_OUTPUT_DIR / 'var_confidence_audit.pdf')
